In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base_path = "/content/drive/MyDrive/AI-Powered Hospitality Revenue Optimization/Dataset/"

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv(base_path + "hotel_bookings_cleaned.csv")

In [5]:
monthly = (
    df.groupby([
        "arrival_date_year",
        "arrival_date_month",
        "hotel"
    ])
    .agg(
        bookings=("hotel", "size"),
        cancellations=("is_canceled", "sum"),
        avg_adr=("adr", "mean"),
        revenue=("estimated_revenue", "sum")
    )
    .reset_index()
)

monthly["cancellation_rate"] = (
    monthly["cancellations"] /
    monthly["bookings"] * 100
)

In [6]:
hotel_avg_bookings = (
    monthly.groupby("hotel")["bookings"]
    .mean()
    .to_dict()
)

hotel_avg_bookings

{'City Hotel': 2049.0, 'Resort Hotel': 1306.0}

In [7]:
def recommendation(row):

    avg_bookings = hotel_avg_bookings[row["hotel"]]

    if row["cancellation_rate"] >= 30:
        return "High cancellation risk – review cancellation and deposit strategy"

    elif (
        row["bookings"] > avg_bookings
        and row["avg_adr"] < 80
    ):
        return "High demand with low ADR – review pricing"

    elif row["bookings"] < avg_bookings:
        return "Lower demand – consider targeted promotions"

    else:
        return "Normal"

In [8]:
monthly["recommendation"] = monthly.apply(
    recommendation,
    axis=1
)

In [9]:
monthly["issue"] = monthly["recommendation"].map({
    "High cancellation risk – review cancellation and deposit strategy":
        "High cancellation rate",

    "High demand with low ADR – review pricing":
        "High demand with low ADR",

    "Lower demand – consider targeted promotions":
        "Lower demand",

    "Normal":
        "Normal"
})

In [10]:
opportunities = monthly[
    monthly["recommendation"] != "Normal"
].copy()

In [11]:
opportunities = opportunities[
    [
        "arrival_date_year",
        "arrival_date_month",
        "hotel",
        "issue",
        "bookings",
        "cancellation_rate",
        "avg_adr",
        "revenue",
        "recommendation"
    ]
].sort_values(
    ["hotel", "arrival_date_year", "bookings"],
    ascending=[True, True, False]
)

opportunities

,arrival_date_year,arrival_date_month,hotel,issue,bookings,cancellation_rate,avg_adr,revenue,recommendation
10,2015,September,City Hotel,Lower demand,1667,18.776245,107.116053,520787.11,Lower demand – consider targeted promotions
8,2015,October,City Hotel,Lower demand,1555,18.585209,95.333093,429059.68,Lower demand – consider targeted promotions
0,2015,August,City Hotel,Lower demand,1101,20.799273,83.433170,273750.45,Lower demand – consider targeted promotions
2,2015,December,City Hotel,Lower demand,1015,21.674877,83.080069,266437.80,Lower demand – consider targeted promotions
6,2015,November,City Hotel,Lower demand,795,17.232704,74.078579,181539.73,Lower demand – consider targeted promotions
4,2015,July,City Hotel,High cancellation rate,393,59.033079,67.153028,93074.80,High cancellation risk – review cancellation a...
14,2016,August,City Hotel,High cancellation rate,2808,32.407407,124.510385,1165076.17,High cancellation risk – review cancellation a...
32,2016,October,City Hotel,High cancellation rate,2642,31.718395,114.796734,910651.99,High cancellation risk – review cancellation a...
12,2016,April,City Hotel,High cancellation rate,2410,31.037344,103.940494,773615.98,High cancellation risk – review cancellation a...
16,2016,December,City Hotel,High cancellation rate,1983,38.275340,98.387100,665472.41,High cancellation risk – review cancellation a...


In [13]:
opportunities.to_csv(base_path + "business_recommendation.csv", index=False)